In [1]:
import torch
import torch.nn as nn

class Linear(nn.Module):
    def __init__(self,d_in,d_out,device=None,dtype=None):
        super().__init__()
        self.w=nn.Parameter(torch.empty(d_out,d_in,device=device,dtype=dtype))
        mean,std=0,(2/(d_in+d_out))**0.5
        nn.init.trunc_normal_(self.w,mean=mean,std=std,a=-3*std,b=3*std)
    
    def forward(self,x):
        y=x @ self.w.T
        return y

In [2]:
m=Linear(64,256)
x=torch.randn(32,54,64)
output=m(x)
print(output.shape)


torch.Size([32, 54, 256])


In [3]:
class Embedding(nn.Module):
    def __init__(self,vocab_size,embed_dim,device=None,dtype=None):
        super().__init__()
        self.embed=nn.Parameter(torch.empty(vocab_size,embed_dim,device=device,dtype=dtype))
        nn.init.trunc_normal_(self.embed,mean=0,std=1,a=-3,b=3)

    def forward(self,token_id):
        return self.embed[token_id]



In [4]:
E=Embedding(1000,256)
token_id=torch.tensor([[1,2,3],[4,5,6]])
output=E(token_id)
print(output.shape)


torch.Size([2, 3, 256])


In [5]:
class RMSNorm(nn.Module):
    def __init__(self,embed_dim,eps=1e-5,device=None,dtype=None):
        super().__init__()
        self.embed_dim=embed_dim
        self.eps=eps
        self.g=nn.Parameter(torch.ones(embed_dim,device=device,dtype=dtype))

    def forward(self,x):
        in_dtype=x.dtype
        x=x.to(torch.float32)

        rms=torch.sqrt(x.square().sum(dim=-1)/self.embed_dim+self.eps)

        res=x/rms*self.g

        return res

In [6]:
x=torch.tensor([
    [12,23,43],
    [-100,290,350],
    [123,124,90]
]).float()

norm=RMSNorm(embed_dim=3)

print(x)

output=norm(x)

print(output)

tensor([[  12.,   23.,   43.],
        [-100.,  290.,  350.],
        [ 123.,  124.,   90.]])
tensor([[ 0.4139,  0.0856,  0.3791],
        [-3.4490,  1.0793,  3.0854],
        [ 4.2422,  0.4615,  0.7934]], grad_fn=<MulBackward0>)


In [ ]:
from einops import rearrange

class Rope(nn.Module):
    def __init__(self,theta,d_k,max_seq_len,device=None,dtype=None):
        super().__init__()
        pos=torch.arange(max_seq_len,device=device) #(max_seq_len,)
        #生成每组向量对应的维度下标
        dim_indices=torch.arange(0,d_k,2,device=device) #(d_k//2,)
        #第 0 组：维度 0、1
        #第 1 组：维度 2、3
        #第 2 组：维度 4、5
        #第 3 组：维度 6、7

        #计算不同维度组的旋转速度
        freqs=theta ** -(dim_indices/d_k)  #(d_k//2,)

        #利用广播生成角度表
        #假设pos=[0,1,2,3]  freqs=[1,0.1,0.01,0.001]
        angle=pos[:,None] * freqs[None,:] # None给Tensor增加一个长度为1的维度
        # pos:(4,)->(4,1),freqs:(4,)->(1,4)
        '''
        angle =
[
  [0*1, 0*0.1, 0*0.01, 0*0.001],
  [1*1, 1*0.1, 1*0.01, 1*0.001],
  [2*1, 2*0.1, 2*0.01, 2*0.001],
  [3*1, 3*0.1, 3*0.01, 3*0.001],
]
'''
        #angle[pos,dim] 位置pos,第dim个二维维度组的旋转角度

        cos_table=torch.cos(angle)  
        sin_table=torch.sin(angle)  #(max_seq_len,d_k//2)

        self.register_buffer("cos_table",cos_table)
        self.register_buffer("sin_table",sin_table)

    
    def forward(self,x,token_pos):
        # x->(...,seq_len,d_k)
        #token_pos ->(...,seq_len)
        cos=self.cos_table[token_pos]
        sin=self.sin_table[token_pos]

        x1,x2=rearrange(x,"... (a b) -> ... a b",b=2).unbind(dim=-1) #把最后一维度拆掉，分成多个tensor

        x1_rot=x1*cos-x2*sin
        x2_rot=x1*sin+x2*cos   # 旋转公式: b' = a*sin + b*cos (a=x1, b=x2)

        x_rot=torch.stack([x1_rot,x2_rot],dim=-1)  
        x_rot=rearrange(x_rot,"... a b -> ... (a b)")

        return x_rot


In [9]:
x=torch.tensor([
    [
        [1,2,3,4],
        [5,6,7,8],
        [9,198,23,43],
        [-20,-43,56,89],
    ],
    [
        [32,43,12,90],
        [-20,98,76,0],
        [1,3,5,3],
        [2,5,7,8]
    ]
]).float()
print(x.shape)

token_pos=torch.tensor([
    [0,1,2,3],
    [4,5,6,7]
])
rope=Rope(theta=10000.0,d_k=4,max_seq_len=1024)
output=rope(x,token_pos)

print(f"输入张量x: {x}")

print(f"输出张量: {output}")

print(output.shape)


torch.Size([2, 4, 4])
输入张量x: tensor([[[  1.,   2.,   3.,   4.],
         [  5.,   6.,   7.,   8.],
         [  9., 198.,  23.,  43.],
         [-20., -43.,  56.,  89.]],

        [[ 32.,  43.,  12.,  90.],
         [-20.,  98.,  76.,   0.],
         [  1.,   3.,   5.,   3.],
         [  2.,   5.,   7.,   8.]]])
输出张量: tensor([[[ 1.0000e+00,  1.0000e+00,  3.0000e+00,  3.0000e+00],
         [-2.3473e+00,  7.7503e+00,  6.9197e+00,  7.0796e+00],
         [-1.8379e+02,  1.7630e+02,  2.2135e+01,  2.3855e+01],
         [ 2.5868e+01,  1.3732e+01,  5.3305e+01,  5.8644e+01]],

        [[ 1.1626e+01, -5.3459e+01,  8.3914e+00,  1.5589e+01],
         [ 8.8301e+01, -9.9648e+01,  7.5905e+01,  7.5905e+01],
         [ 1.7984e+00,  1.2192e-01,  4.8111e+00,  5.1709e+00],
         [-1.7771e+00,  4.7927e+00,  6.4233e+00,  7.5424e+00]]])
torch.Size([2, 4, 4])
